# FRI-BCI large GPU campaign
Six source runs, 810 target trajectories, 2,430 scored checkpoints. Select your GPU runtime first. This notebook performs development on target R08, never target R12. Full details: docs/GPU_CAMPAIGN_2026-11-12.md.


In [ ]:
from pathlib import Path
import subprocess, sys, os, hashlib
REPO = Path('/content/FRI-BCI-GPU-Campaign')
URL = 'https://github.com/Mohnclizzy/FRI-BCI-ML-RL-Decoder-Personal.git'
REF = 'codex/robot-learning-design'
if not REPO.exists():
    subprocess.run(['git', 'clone', '--branch', REF, URL, str(REPO)], check=True)
os.chdir(REPO)
print('Checkout:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())
EXPECTED = {'eeg_decoder/gpu_campaign.py': 'dc7b0fd3c1a8ed6f7cbfb06d58902f7f57340064bf4dba6a3873f89ef8679bae', 'configs/gpu_campaign.json': 'c8cd084d02e7fa0f540a4f11df31da28dc1c0b93367d5a290cdd69c850f0e66b', 'configs/gpu_research.json': '1da8f557732c094968cc5b542662377b06c177198077feb880e5123385ddd19d', 'eeg_decoder/gpu_pipeline.py': '8ea200dd7a16d81ee56da12609b64222368f051878bcbf60f70a22e2ceb0aaa1', 'eeg_decoder/neural.py': '6d495b1673317a3a1bdad3c2795ac5d1257e98fd3f7b440a58aa6ff07abb406a', 'eeg_decoder/alignment.py': '043dedee70d0f62da3410938a2a837585678e8fe1134179d290613077bee5c1b', 'eeg_decoder/baseline.py': 'b65f3fe48c083daba6fb631af52a239954debc7e4bcad0c299fbe4a0300eca1d', 'eeg_decoder/risk_gate.py': '8c9ed23def0adf6f9d5226f59482ff4d98af16c1659904b6bb786aff22ecaa68'}
for name, expected in EXPECTED.items():
    assert hashlib.sha256(Path(name).read_text(encoding='utf-8').encode()).hexdigest() == expected, f'{name}: version mismatch; use matching campaign checkout'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'], check=True)
import torch
assert torch.cuda.is_available(), 'Select a GPU runtime before training'
print(torch.__version__, torch.cuda.get_device_name(0))


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
WORK = '/content/drive/MyDrive/FRI_BCI/gpu_campaign_2026_11_12'
CACHE = '/content/fri_campaign_cache'
Path(WORK).mkdir(parents=True, exist_ok=True)
subprocess.run([sys.executable, '-m', 'pip', 'freeze'], stdout=open(Path(WORK)/'environment.txt', 'w'), check=True)
subprocess.run([sys.executable, '-m', 'eeg_decoder.gpu_campaign', 'plan'], check=True)


In [ ]:
# None runs all six shards sequentially. Use a distinct 0..5 per separate host.
SHARD = None
command = [sys.executable, '-u', '-m', 'eeg_decoder.gpu_campaign', 'run', '--work-dir', WORK, '--cache-dir', CACHE]
if SHARD is not None:
    command += ['--shard', str(SHARD)]
log = Path(WORK) / f'launch_{SHARD}.log'
with log.open('a') as handle:
    with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
        for line in process.stdout:
            print(line, end='')
            handle.write(line)
            handle.flush()
        code = process.wait()
        if code:
            raise RuntimeError(f'Campaign failed ({code}); inspect {log}')


In [ ]:
subprocess.run([sys.executable, '-m', 'eeg_decoder.gpu_campaign', 'summarize', '--work-dir', WORK], check=True)
print(Path(WORK, 'development_summary.json').read_text())
